# 05_6 — Live Scoring con LSTM

Scoring de mercados activos usando `price_sequence_lstm`. Este notebook no reentrena: lee artefactos en `data/models/price_sequence_lstm/` y scores generados por `python -m src.scoring.scorer --config config/config.yaml --all-models`.


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config

cfg = load_config(ROOT / 'config' / 'config.yaml')
MODEL_NAME = 'price_sequence_lstm'
MODEL_DIR = ROOT / 'data' / 'models' / MODEL_NAME
REGISTRY = ROOT / 'data' / 'models' / 'registry'
PALETTE = ['#2A9D8F', '#264653', '#E76F51', '#8D99AE']


## 1. Métricas offline guardadas


In [ ]:
with open(MODEL_DIR / 'run_config.json') as f:
    run_config = json.load(f)
with open(MODEL_DIR / 'test_metrics.json') as f:
    test_metrics = json.load(f)
with open(REGISTRY / 'live_scoring_summary.json') as f:
    live_summary = json.load(f).get(MODEL_NAME, {})

tm = test_metrics['test']
print('Modelo:')
display(pd.Series(run_config['model']))
print()
print('Offline test:')
print(f"Brier LSTM / mercado    : {tm['calibrated_metrics']['brier']:.4f} / {tm['market_baseline_metrics']['brier']:.4f}")
print(f"Log-loss LSTM / mercado : {tm['calibrated_metrics']['log_loss']:.4f} / {tm['market_baseline_metrics']['log_loss']:.4f}")
print(f"Top-K realized PnL      : {tm['ev_metrics']['top_k_avg_realized_pnl']:.4f}")
print(f"Top-K hit rate          : {tm['ev_metrics']['top_k_hit_rate']:.3f}")


## 2. Scores live


In [ ]:
df_all = pd.read_csv(REGISTRY / 'live_scores.csv')
df_scores = df_all[df_all['model_name'] == MODEL_NAME].copy()

print(f'Mercados activos scored: {len(df_scores):,}')
print('Resumen live:')
display(pd.Series(live_summary.get('signals', {})))

display(df_scores.head(10))


## 3. Distribución de señales y scores


In [ ]:
signal_counts = df_scores['signal'].value_counts().reindex(['STRONG BUY', 'BUY', 'HOLD'], fill_value=0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(signal_counts.index, signal_counts.values, color=PALETTE[:3])
axes[0].set_title('Señales live')
axes[0].tick_params(axis='x', rotation=20)

axes[1].hist(df_scores['p_yes_calibrated'], bins=40, color=PALETTE[0], alpha=0.85)
axes[1].set_title('p_yes_calibrated')

axes[2].hist(df_scores['ev_per_share'], bins=40, color=PALETTE[2], alpha=0.85)
axes[2].axvline(cfg['scoring']['buy_ev_threshold'], color='gray', ls='--')
axes[2].set_title('EV por share')
plt.tight_layout()


## 4. Top mercados por EV

Estos no se deben presentar como recomendación final. El backtest mostró que el LSTM sobreestima EV en el Top-K.


In [ ]:
cols = ['signal', 'question', 'price_yes', 'p_yes_calibrated', 'ev_per_share', 'expected_roi_capped', 'liquidity', 'volume_24h', 'spread']
top_ev = df_scores.sort_values('ev_per_share', ascending=False).head(20)
display(top_ev[cols])


## 5. Comparación live contra HistBoost y CatBoost


In [ ]:
models = ['price_sequence_lstm', 'hist_gradient_boosting', 'catboost_residual']
rows = []
for model in models:
    sub = df_all[df_all['model_name'] == model]
    rows.append({
        'model': model,
        'rows': len(sub),
        'strong_buy': int((sub['signal'] == 'STRONG BUY').sum()),
        'buy': int((sub['signal'] == 'BUY').sum()),
        'hold': int((sub['signal'] == 'HOLD').sum()),
        'median_ev': float(sub['ev_per_share'].median()) if len(sub) else np.nan,
        'p90_ev': float(sub['ev_per_share'].quantile(0.90)) if len(sub) else np.nan,
    })
live_compare = pd.DataFrame(rows)
display(live_compare)

fig, ax = plt.subplots(figsize=(9, 4))
live_compare.set_index('model')[['strong_buy', 'buy', 'hold']].plot.bar(stacked=True, ax=ax, color=PALETTE[:3])
ax.set_title('Distribución live de señales')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()


## 6. Lectura operativa

El LSTM genera muchas señales parecidas al GRU/baseline, pero el test offline indica que esas señales no fueron rentables. Para la entrega, úsalo como contraste: captura patrones de secuencia, pero no produce edge económico robusto.
